# 04 · Evaluation framework and baselines

Before building any sophisticated model, this step fixes **how models are judged** and sets the **bar to beat**:

1. a walk-forward backtest that never lets a model see the future,
2. loss functions suited to volatility,
3. five baseline forecasters that need no estimation.

Every later model (GARCH, HAR, LightGBM, neural networks) goes through exactly the same procedure and is stored in the same tables.

**Run first** (from the project root):
```bash
python -m src.data.setup_db            # adds the forecasts and model_scores tables
python -m src.models.run_baselines
```

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

from src.db import get_engine
from src.evaluation.metrics import log_squared_error, qlike
from src.evaluation.splits import walk_forward_folds
from src.viz import AQUA, BLUE, BLUE_LIGHT, CRISES, GRID, INK, INK_2, MUTED, ORANGE, save, set_style, shade_crises

set_style()
engine = get_engine()
pd.set_option("display.width", 180)
pd.set_option("display.float_format", "{:,.3f}".format)

LABELS = {"naive": "Naive (last value)", "monthly_average": "Monthly average", "historical_mean": "Historical mean",
          "ewma": "EWMA of range-based variance", "riskmetrics": "RiskMetrics (EWMA of squared returns)"}

## 1. Walk-forward backtest

A forecasting model must be judged on data it has never seen, and the order of time must be respected: a model used in 2015 could only have been trained on data up to 2014.

For each **test year** from 2006 to 2026, the model is trained on all earlier data and forecasts every day of that year. The training window then grows by one year (*expanding window*). This gives 21 years of genuinely out-of-sample forecasts, covering the 2008 crisis, the euro debt crisis, COVID-19 and the 2022 shock.

**Embargo.** A row dated *t* has a target computed from days *t+1 … t+h*. The last *h* trading days before each test year are therefore removed from the training set; otherwise their targets would overlap the test period.

In [ ]:
dates = pd.read_sql("SELECT DISTINCT date FROM features ORDER BY date", engine, parse_dates=["date"])["date"]
folds = walk_forward_folds(dates, embargo=22)
shown = [f for f in folds if f.test_year in (2006, 2007, 2008, 2015, 2020, 2026)]

fig, ax = plt.subplots(figsize=(9, 3.2))
for row, fold in enumerate(shown):
    ax.barh(row, (fold.train_end - dates.min()).days, left=mdates.date2num(dates.min()), color=MUTED, height=0.55)
    ax.barh(row, (fold.test_end - fold.test_start).days, left=mdates.date2num(fold.test_start), color=BLUE, height=0.55)
    ax.text(mdates.date2num(fold.test_end) + 120, row, f"test {fold.test_year}", va="center", color=INK_2, fontsize=9)
ax.set_yticks([])
ax.invert_yaxis()
ax.xaxis_date()
ax.xaxis.set_major_locator(mdates.YearLocator(4))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_xlim(mdates.date2num(dates.min()) - 200, mdates.date2num(dates.max()) + 1100)
ax.grid(axis="y", visible=False)
ax.set_title("Walk-forward evaluation: train on the past (grey), forecast the next year (blue)")
fig.tight_layout()
save(fig, "05_walk_forward")
plt.show()

first, last = folds[0], folds[-1]
print(f"{len(folds)} folds. First: train to {first.train_end:%Y-%m-%d}, test {first.test_start:%Y-%m-%d} to {first.test_end:%Y-%m-%d}")
print(f"          Last:  train to {last.train_end:%Y-%m-%d}, test {last.test_start:%Y-%m-%d} to {last.test_end:%Y-%m-%d}")

## 2. Loss functions

Forecasts and outcomes are compared as **variances** (average daily variance over the horizon).

| Loss | Formula | Why |
|---|---|---|
| **QLIKE** | actual/forecast − ln(actual/forecast) − 1 | Standard in the volatility literature. Robust to the noise in the realised-variance measure, depends only on the ratio (calm and turbulent periods count equally) and penalises **under**-prediction of risk more than over-prediction |
| **log MSE** | (ln actual − ln forecast)² | Squared error on the log scale, symmetric |
| **MAE (vol points)** | \|actual − forecast\| in annualised % | Easy to communicate |

QLIKE is the primary criterion. Its asymmetry matches risk management practice: a forecast that is half the realised variance is more costly than one that is double.

In [ ]:
ratio = np.linspace(0.2, 5, 400)               # forecast / actual
fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(ratio, qlike(1, ratio), color=BLUE, lw=2, label="QLIKE")
ax.plot(ratio, log_squared_error(1, ratio), color=ORANGE, lw=2, label="log MSE")
ax.axvline(1, color=MUTED, lw=1)
ax.set_xscale("log")
ax.set_xticks([0.25, 0.5, 1, 2, 4], ["4x too low", "2x too low", "exact", "2x too high", "4x too high"])
ax.set_ylim(0, 2.5)
ax.set_title("Loss as a function of the forecast error")
ax.set_xlabel("forecast relative to the realised variance")
ax.legend(loc="upper center")
fig.tight_layout()
save(fig, "05_loss_functions")
plt.show()
print(f"QLIKE when the forecast is half the outcome: {qlike(1, 0.5):.3f}; when it is double: {qlike(1, 2):.3f}")

## 3. The baselines

| Model | Forecast of the average variance over the next *h* days |
|---|---|
| **Naive** | The variance observed over the last *h* days |
| **Monthly average** | The average of the last 22 days |
| **Historical mean** | The series' average over all history up to day *t* |
| **EWMA** | Exponentially weighted average of the daily range-based variance (λ = 0.94) |
| **RiskMetrics** | The same on squared close-to-close returns: the classic J.P. Morgan model |

None has estimated parameters. EWMA and RiskMetrics differ only in the variance measure they are fed, which isolates the value of the range-based measure built in Step 4.

## 4. Results

Scores are computed in SQL ([`sql/score_models.sql`](../sql/score_models.sql)) on the **common sample**: the observations for which every model has a forecast.

In [ ]:
overall = pd.read_sql("SELECT * FROM model_scores_overall", engine)
overall["model_label"] = overall["model"].map(LABELS)
print(f"Forecasts per model and horizon: about {overall['n_forecasts'].median():,.0f}")

def table(metric):
    t = overall.pivot(index="model_label", columns="horizon", values=metric)
    t.columns = [f"{h} day" + ("s" if h > 1 else "") for h in t.columns]
    return t.sort_values(t.columns[1])

print("\nQLIKE (lower is better)")
display(table("qlike"))
print("MAE in annualised volatility points")
display(table("mae_vol"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3), sharex=True)
order = table("qlike").index[::-1]
for ax, h in zip(axes, [1, 5, 22]):
    scores = overall[overall["horizon"] == h].set_index("model_label").loc[order, "qlike"]
    ax.barh(scores.index, scores.values, color=BLUE, height=0.6)
    for y, v in enumerate(scores.values):
        ax.text(v + 0.012, y, f"{v:.3f}", va="center", color=INK_2, fontsize=9)
    ax.set_title(f"{h}-day horizon", fontsize=11)
    ax.grid(axis="y", visible=False)
    if h != 1:
        ax.set_yticklabels([])
axes[0].set_xlim(0, overall["qlike"].max() * 1.18)
fig.suptitle("Out-of-sample QLIKE of the baselines, 2006-2026 (lower is better)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
save(fig, "05_baseline_scores")
plt.show()

**Reading the results.**

* **EWMA is the baseline to beat** on QLIKE at every horizon. On the mean absolute error at one day, EWMA and RiskMetrics are essentially tied.
* **The historical mean is the worst** beyond one day: volatility is too persistent to be forecast by its long-run average. At one day it is the naive forecast that fails, because a single day's variance is too noisy to be repeated as a forecast.
* **Measurement matters:** EWMA and RiskMetrics use the same formula; feeding it the range-based variance instead of squared returns lowers QLIKE at every horizon.
* **Longer horizons have lower losses** because the target, an average over more days, is less noisy.

In [ ]:
value_of_range = overall.pivot(index="horizon", columns="model", values="qlike")
(1 - value_of_range["ewma"] / value_of_range["riskmetrics"]).map("{:.1%}".format).to_frame("QLIKE reduction: EWMA vs RiskMetrics").T

## 5. Stability over time and across regimes

A single average can hide failures in crises. The chart shows the 5-day QLIKE by test year.

In [ ]:
by_year = pd.read_sql("""
    SELECT model, horizon, test_year, sum(qlike * n) / sum(n) AS qlike
    FROM model_scores GROUP BY model, horizon, test_year""", engine)
y5 = by_year[by_year["horizon"] == 5].pivot(index="test_year", columns="model", values="qlike")

fig, ax = plt.subplots(figsize=(9, 3.6))
for model, color in [("historical_mean", ORANGE), ("naive", AQUA), ("ewma", BLUE)]:
    ax.plot(y5.index, y5[model], color=color, lw=2, marker="o", ms=4, label=LABELS[model])
ax.set_title("5-day QLIKE by test year")
ax.set_xticks(y5.index[::2])
ax.grid(axis="x", visible=False)
ax.legend(loc="upper right", ncols=3)
ax.set_ylim(0, y5[["historical_mean", "naive", "ewma"]].max().max() * 1.2)
fig.tight_layout()
save(fig, "05_scores_by_year")
plt.show()

In [ ]:
# Calm vs stress periods, computed in SQL on the common sample (5-day horizon)
stress = " OR ".join(f"f.date BETWEEN '{start}' AND '{end}'" for start, end in CRISES.values())
regimes = pd.read_sql(f"""
    WITH joined AS (
        SELECT f.model, f.pred_var::float8 AS forecast, exp(x.target_5d) AS actual,
               CASE WHEN {stress} THEN 'stress periods' ELSE 'calm periods' END AS regime,
               count(*) OVER (PARTITION BY f.ticker, f.date) AS models_with_forecast
        FROM forecasts_long f JOIN features x USING (ticker, date)
        WHERE f.horizon = 5
    )
    SELECT model, regime, count(*) AS n, avg(actual / forecast - ln(actual / forecast) - 1) AS qlike
    FROM joined
    WHERE actual IS NOT NULL AND models_with_forecast = (SELECT count(*) FROM models)
    GROUP BY model, regime""", engine)
regime_table = regimes.assign(model=regimes["model"].map(LABELS)).pivot(index="model", columns="regime", values="qlike")
regime_table["ratio stress / calm"] = regime_table["stress periods"] / regime_table["calm periods"]
print("Share of forecasts in stress periods:", f"{regimes.groupby('regime')['n'].sum().pipe(lambda s: s['stress periods'] / s.sum()):.0%}")
regime_table.sort_values("calm periods")

**Reading the table.** The ranking is not the same in both regimes:

* in **calm periods** EWMA is clearly the best baseline;
* in **stress periods** the naive forecast, which reacts fastest, has the lowest QLIKE: when volatility jumps, an average over a long window is slow to catch up;
* the historical mean deteriorates the most, its loss is about 1.5 times higher in stress periods.

A good model should therefore be judged on both regimes, not only on the average. Step 9 repeats this split for all models.

In [ ]:
# Stocks vs indices (5-day horizon)
by_type = pd.read_sql("""
    SELECT s.model, u.asset_type, sum(s.qlike * s.n) / sum(s.n) AS qlike
    FROM model_scores s JOIN universe u USING (ticker)
    WHERE s.horizon = 5 GROUP BY s.model, u.asset_type""", engine)
by_type.assign(model=by_type["model"].map(LABELS)).pivot(index="model", columns="asset_type", values="qlike").sort_values("stock")

## 6. What a forecast looks like

Five-day-ahead forecasts for the CAC 40 around the COVID-19 crash. Each point is a forecast made on that day for the following five trading days, next to what actually happened.

In [ ]:
example = pd.read_sql("""
    SELECT model, date, pred_vol_pct, actual_vol_pct FROM forecast_vs_actual
    WHERE ticker = '^FCHI' AND horizon = 5 AND date BETWEEN '2020-01-15' AND '2020-07-31'
      AND model IN ('ewma', 'historical_mean')""", engine, parse_dates=["date"])
actual = example[example["model"] == "ewma"].set_index("date")["actual_vol_pct"].sort_index()
pred = example.pivot(index="date", columns="model", values="pred_vol_pct").sort_index()

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(actual.index, actual, color=MUTED, lw=1.5, label="Realised over the next 5 days")
ax.plot(pred.index, pred["ewma"], color=BLUE, lw=2, label="EWMA forecast")
ax.plot(pred.index, pred["historical_mean"], color=ORANGE, lw=2, label="Historical mean forecast")
ax.set_title("CAC 40: 5-day volatility forecasts during the COVID-19 crash (annualised, %)")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.legend(loc="upper right")
fig.tight_layout()
save(fig, "05_forecast_example")
plt.show()

The picture shows the limits of simple rules. The historical mean barely moves. EWMA adapts, but only **after** volatility has risen: it is a backward-looking average, so it lags at the start of the crash and stays too high once markets calm down. Reacting faster on the way up and reverting faster on the way down is what the models of the next steps must achieve.

## Summary

* **Framework:** 21 yearly walk-forward folds with an embargo equal to the horizon; QLIKE as the main loss; all models compared on a common sample; forecasts and scores stored in PostgreSQL (`forecasts`, `model_scores`).
* **Bar to beat:** EWMA of the range-based variance.
* **First result:** the same EWMA formula is clearly better with the range-based variance than with squared returns, which confirms the choice made in Step 4.

Next: Step 6 adds the econometric models (GARCH, GJR-GARCH, HAR).